# 03 · Data Wrangling: limpieza y creación de la variable objetivo

**Objetivo:** explorar el dataset de la API y transformar el resultado del aterrizaje (`Outcome`) en una
etiqueta binaria `Class`:

* `1` → la primera etapa **aterrizó** con éxito.
* `0` → no aterrizó (fallo o no se intentó).

| Código en `Outcome` | Significado |
|---|---|
| `True ASDS` / `False ASDS` | Aterrizaje exitoso / fallido en barcaza autónoma (*drone ship*) |
| `True RTLS` / `False RTLS` | Aterrizaje exitoso / fallido en plataforma terrestre (*Return To Launch Site*) |
| `True Ocean` / `False Ocean` | Descenso controlado exitoso / fallido sobre el océano |
| `None None` / `None ASDS` | No hubo intento de aterrizaje |

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
IMAGES_DIR = Path("../images")
for folder in (DATA_DIR, RESULTS_DIR, IMAGES_DIR):
    folder.mkdir(exist_ok=True)

# Repositorio público de datos del curso IBM Applied Data Science Capstone
IBM = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork"


def load_csv(name: str, url: str) -> pd.DataFrame:
    """Lee data/<name> si ya existe; si no, lo descarga desde `url` y lo guarda como caché local."""
    local = DATA_DIR / name
    if local.exists():
        print(f"Leyendo {local}")
        return pd.read_csv(local)
    print(f"Descargando {url}")
    df = pd.read_csv(url)
    df.to_csv(local, index=False)
    return df

In [2]:
import json

df = load_csv("dataset_part_1.csv", f"{IBM}/datasets/dataset_part_1.csv")
print(df.shape)
df.head()

Leyendo ../data/dataset_part_1.csv
(90, 17)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


## 1. Valores faltantes (%) y tipos de dato

In [3]:
(df.isnull().sum() / len(df) * 100).round(2)

FlightNumber       0.00
Date               0.00
BoosterVersion     0.00
PayloadMass        0.00
Orbit              0.00
LaunchSite         0.00
Outcome            0.00
Flights            0.00
GridFins           0.00
Reused             0.00
Legs               0.00
LandingPad        28.89
Block              0.00
ReusedCount        0.00
Serial             0.00
Longitude          0.00
Latitude           0.00
dtype: float64

In [4]:
df.dtypes

FlightNumber        int64
Date                  str
BoosterVersion        str
PayloadMass       float64
Orbit                 str
LaunchSite            str
Outcome               str
Flights             int64
GridFins             bool
Reused               bool
Legs                 bool
LandingPad            str
Block             float64
ReusedCount         int64
Serial                str
Longitude         float64
Latitude          float64
dtype: object

## 2. Lanzamientos por sitio

In [5]:
df["LaunchSite"].value_counts()

LaunchSite
CCAFS SLC 40    55
KSC LC 39A      22
VAFB SLC 4E     13
Name: count, dtype: int64

## 3. Lanzamientos por órbita

In [6]:
df["Orbit"].value_counts()

Orbit
GTO      27
ISS      21
VLEO     14
PO        9
LEO       7
SSO       5
MEO       3
ES-L1     1
HEO       1
SO        1
GEO       1
Name: count, dtype: int64

## 4. Resultados de aterrizaje

In [7]:
landing_outcomes = df["Outcome"].value_counts()
landing_outcomes

Outcome
True ASDS      41
None None      19
True RTLS      14
False ASDS      6
True Ocean      5
False Ocean     2
None ASDS       2
False RTLS      1
Name: count, dtype: int64

In [8]:
for i, outcome in enumerate(landing_outcomes.keys()):
    print(i, outcome)

0 True ASDS
1 None None
2 True RTLS
3 False ASDS
4 True Ocean
5 False Ocean
6 None ASDS
7 False RTLS


Los resultados que **no** son un aterrizaje exitoso:

In [9]:
bad_outcomes = {outcome for outcome in landing_outcomes.keys() if not outcome.startswith("True")}
bad_outcomes

{'False ASDS', 'False Ocean', 'False RTLS', 'None ASDS', 'None None'}

## 5. Variable objetivo `Class`

In [10]:
landing_class = [0 if outcome in bad_outcomes else 1 for outcome in df["Outcome"]]
df["Class"] = landing_class
df[["Outcome", "Class"]].head(8)

,Outcome,Class
0,None None,0
1,None None,0
2,None None,0
3,False Ocean,0
4,None None,0
5,None None,0
6,True Ocean,1
7,True Ocean,1


In [11]:
success_rate = df["Class"].mean()
print(f"Tasa global de aterrizajes exitosos: {success_rate:.2%}")

Tasa global de aterrizajes exitosos: 66.67%


In [12]:
df.to_csv(DATA_DIR / "dataset_part_2.csv", index=False)

summary = {
    "rows": int(len(df)),
    "success_rate": round(float(success_rate), 4),
    "launches_by_site": df["LaunchSite"].value_counts().to_dict(),
    "launches_by_orbit": df["Orbit"].value_counts().to_dict(),
    "landing_outcomes": {k: int(v) for k, v in landing_outcomes.items()},
    "bad_outcomes": sorted(bad_outcomes),
}
(RESULTS_DIR / "data_wrangling.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))
summary

{'rows': 90,
 'success_rate': 0.6667,
 'launches_by_site': {'CCAFS SLC 40': 55, 'KSC LC 39A': 22, 'VAFB SLC 4E': 13},
 'launches_by_orbit': {'GTO': 27,
  'ISS': 21,
  'VLEO': 14,
  'PO': 9,
  'LEO': 7,
  'SSO': 5,
  'MEO': 3,
  'ES-L1': 1,
  'HEO': 1,
  'SO': 1,
  'GEO': 1},
 'landing_outcomes': {'True ASDS': 41,
  'None None': 19,
  'True RTLS': 14,
  'False ASDS': 6,
  'True Ocean': 5,
  'False Ocean': 2,
  'None ASDS': 2,
  'False RTLS': 1},
 'bad_outcomes': ['False ASDS',
  'False Ocean',
  'False RTLS',
  'None ASDS',
  'None None']}

## Conclusión

El dataset queda con la etiqueta `Class`, que es lo que intentaremos **predecir** en el notebook 07.
La tasa de éxito global sirve como línea base: un modelo útil debe superar a "predecir siempre aterriza".